# 1D BSPF: a fit, a residual, and pure functions

Install from the repository root: `python -m pip install -e '.[host,notebook,test]'`.
Select that environment as your notebook kernel. No source-path modification is needed.

We use $f(x)=x^3+2x$, so $f'(x)=3x^2+2$ and $\int_{-1}^{1}f(x)dx=0$.
The plan owns immutable geometry and factorization arrays. All operations below
reuse it; JIT and differentiation belong to the caller.


In [ ]:
import pybspf.calculus as bspf_calculus
import pybspf.operators as bspf_operators
import pybspf.plans as bspf_plans

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import pybspf as bspf

x = jnp.linspace(-1.0, 1.0, 65)
plan = bspf_plans.plan_1d(x, degree=5, n_basis=16, boundary_points=7)
f = x**3 + 2*x


## Constrained fit and differentiation

$f=Bc+r$. Endpoint jets constrain $Bc$; they are not a complete PDE boundary
condition for the corrected derivative. A single call shares the fit and FFT
across the requested derivative orders.


In [ ]:
split = jax.jit(bspf_operators.decompose)(plan, f)
derivatives = jax.jit(bspf_operators.derivatives)(plan, f)
error = jnp.max(jnp.abs(derivatives[1] - (3*x**2 + 2)))
print("max derivative error:", float(error))
assert error < 1e-8
assert jnp.max(jnp.abs(split.spline + split.residual - f)) < 1e-12


## Interpolation, integration, and differentiation through the operator

The integral and interpolator use the same spline + Fourier representation.
The loss below is differentiated with respect to input samples, with geometry fixed.


In [ ]:
query = jnp.linspace(-1.0, 1.0, 129)
values = jax.jit(bspf_calculus.interpolate)(plan, f, query)
area = jax.jit(bspf_calculus.integrate)(plan, f)
loss = lambda samples: jnp.sum(bspf_operators.differentiate(plan, samples)**2)
sensitivity = jax.jit(jax.grad(loss))(f)
assert jnp.max(jnp.abs(values - (query**3 + 2*query))) < 1e-10
assert jnp.abs(area) < 1e-10
assert jnp.all(jnp.isfinite(sensitivity))
print("integral:", float(area), "gradient shape:", sensitivity.shape)


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(query, query**3 + 2*query, label="exact")
ax.plot(x, f, ".", label="samples")
ax.plot(query, values, "--", label="BSPF")
ax.set(xlabel="x", ylabel="f(x)")
ax.legend();
